## Code to explore, filter and parse data

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import cellxgene_census
import os

from pathlib import Path

import time


In [2]:
path = '/Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE'

census_version = '2025-11-08'
dataset_id = '77f9d7e9-5675-49c3-abed-ce02f39eef1b'

# Sampling
num_donors = 8
min_cells = 2000
min_types = 8
min_per_group = 10
cap = 100
age_labels = ['< 18',
             '18 - 45',
             '> 45']

# Directories
base_dir = os.path.join(path, 'iha_celltypist')
# base_dir.mkdir(parents = True, exist_ok = True)
os.makedirs(base_dir, exist_ok = True)
parts_dir = os.path.join(base_dir, 'parts')
# parts_dir.mkdir(parents = True, exist_ok = True)
os.makedirs(parts_dir, exist_ok = True)
print('Working directory:', parts_dir)

Working directory: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/parts


In [3]:
with cellxgene_census.open_soma(
    census_version = census_version
) as census:

    datasets = (
        census['census_info']['datasets']
        .read()
        .concat()
        .to_pandas()
    )

atlas_datasets = datasets[
    datasets['collection_id'] == dataset_id
].copy()

print('Number of datasets:', len(atlas_datasets))

display(
    atlas_datasets[
        [
            'dataset_id',
            'dataset_title',
            'dataset_total_cell_count'
        ]
    ].sort_values('dataset_title')
)

Number of datasets: 8


,dataset_id,dataset_title,dataset_total_cell_count
1797,e522d2cd-7927-4e59-a4ed-064009569279,Human Immune Health Atlas,1821725
1468,ed78c6b7-cb40-4ebc-b55f-40bc99ba1b5d,Human Immune Health Atlas B and Plasma cells,160632
1749,e12eb8a9-5e8b-4b59-90c8-77d29a811c00,Human Immune Health Atlas CD4 T cells,743615
1677,b90d4a42-714c-4590-b548-6e75e6c081e8,Human Immune Health Atlas CD8 T cells,406730
1042,dade16db-a38b-41f5-95c8-7ec0238d17fc,Human Immune Health Atlas DCs,23287
1698,24905cd6-203f-4a02-ae7b-cfb4464b2000,Human Immune Health Atlas Monocytes,327919
1463,514329cb-2f22-4e7b-b789-75ea1a1e0067,Human Immune Health Atlas NK cells and ILCs,148605
431,81ffdcd6-3f04-40bc-8bb0-9748f4ffe576,Human Immune Health Atlas Other cells,10937


In [4]:
primary_id = 'e522d2cd-7927-4e59-a4ed-064009569279'

In [5]:
obs_cols = ['soma_joinid',
            'dataset_id',
            'donor_id',
            'cell_type',
            'disease',
            'sex',
            'development_stage',
            'tissue',
            'tissue_general',
            'assay',
            'is_primary_data']

with cellxgene_census.open_soma(census_version = census_version) as census:

    obs = (census['census_data']['homo_sapiens'].obs.read(value_filter = f'dataset_id == "{primary_id}"',column_names = obs_cols).concat().to_pandas())

print('Cells:', len(obs))
print('Donors:', obs.donor_id.nunique())
print('Cell types:', obs.cell_type.nunique())

display(obs.head())

Cells: 1821725
Donors: 108
Cell types: 38


,soma_joinid,dataset_id,donor_id,cell_type,disease,sex,development_stage,tissue,tissue_general,assay,is_primary_data
0,103774024,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,mucosal invariant T cell,normal,female,32-year-old stage,blood,blood,10x 3' v3,False
1,103774025,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,"naive thymus-derived CD4-positive, alpha-beta ...",normal,female,32-year-old stage,blood,blood,10x 3' v3,False
2,103774026,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,"CD14-positive, CD16-negative classical monocyte",normal,female,32-year-old stage,blood,blood,10x 3' v3,False
3,103774027,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,"central memory CD4-positive, alpha-beta T cell",normal,female,32-year-old stage,blood,blood,10x 3' v3,False
4,103774028,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,"CD14-positive, CD16-negative classical monocyte",normal,female,32-year-old stage,blood,blood,10x 3' v3,False


In [6]:
print("Disease:")
display(obs.disease.value_counts(dropna=False))

print("Primary data:")
display(obs.is_primary_data.value_counts(dropna=False))

print("Tissue:")
display(obs.tissue.value_counts(dropna=False))

print("Sex:")
display(obs.sex.value_counts(dropna=False))

print("Age:")
display(obs.development_stage.value_counts(dropna=False))

Disease:


disease
normal                                                                  1053414
cytomegalovirus infection                                                768311
macular degeneration                                                          0
leukoencephalopathy, diffuse hereditary, with spheroids 1                     0
listeriosis                                                                   0
                                                                         ...   
dementia || Alzheimer disease || normal pressure hydrocephalus                0
dementia || Alzheimer disease || schizophrenia                                0
dementia || Alzheimer disease || schizophrenia || Lewy body dementia          0
dementia || Lewy body dementia                                                0
vascular dementia || progressive supranuclear palsy                           0
Name: count, Length: 259, dtype: int64

Primary data:


is_primary_data
False    1557808
True      263917
Name: count, dtype: int64

Tissue:


tissue
blood                          1821725
pigment epithelium of eye            0
perirenal fat                        0
peripheral zone of prostate          0
peripheral region of retina          0
                                ...   
exocrine pancreas                    0
esophagus muscularis mucosa          0
esophagus                            0
esophagogastric junction             0
zone of skin                         0
Name: count, Length: 417, dtype: int64

Sex:


sex
female     989244
male       832481
unknown         0
Name: count, dtype: int64

Age:


development_stage
11-year-old stage    205405
31-year-old stage    137457
55-year-old stage    121653
29-year-old stage    115298
35-year-old stage    102663
                      ...  
39-year-old stage         0
4-month-old stage         0
4-year-old stage          0
40-year-old stage         0
young adult stage         0
Name: count, Length: 194, dtype: int64

In [7]:
with cellxgene_census.open_soma(
    census_version = census_version
) as census:

    schema = census[
        "census_data"
    ]["homo_sapiens"].obs.schema

    available_columns = schema.names

print("Possible age columns:")

for col in available_columns:
    if "age" in col.lower():
        print(col)

Possible age columns:
development_stage
development_stage_ontology_term_id


In [8]:
obs = obs[obs["disease"] == "normal"].copy()

In [9]:
# disease_check = (
#     obs.groupby("donor_id")["disease"]
#     .nunique()
# )

# print(disease_check.value_counts())

disease_check = (
    obs.groupby("donor_id", observed=True)["disease"]
    .nunique()
)

print(disease_check.value_counts())

disease
1    63
Name: count, dtype: int64


In [10]:
display(
    atlas_datasets[
        atlas_datasets.dataset_id == primary_id
    ][
        [
            "dataset_id",
            "dataset_title",
            "dataset_total_cell_count"
        ]
    ]
)

,dataset_id,dataset_title,dataset_total_cell_count
1797,e522d2cd-7927-4e59-a4ed-064009569279,Human Immune Health Atlas,1821725


In [11]:
print(
    obs.groupby(
        ["dataset_id", "is_primary_data"],
        observed=True
    ).size()
)

print(
    "\nUnique donor IDs:",
    obs.donor_id.nunique()
)

print(
    "\nExample donor IDs:"
)

print(
    obs.donor_id.dropna()
    .astype(str)
    .drop_duplicates()
    .head(10)
    .tolist()
)

dataset_id                            is_primary_data
e522d2cd-7927-4e59-a4ed-064009569279  False              846189
                                      True               207225
dtype: int64

Unique donor IDs: 63

Example donor IDs:
['BR1001', 'BR1002', 'BR1003', 'BR1004', 'BR1005', 'BR1006', 'BR2003', 'BR1009', 'BR1011', 'BR1013']


In [12]:
donor_check = (
    obs.groupby("donor_id")
    .agg(
        n_cells=("soma_joinid", "size"),
        n_diseases=("disease", "nunique"),
        n_sexes=("sex", "nunique"),
        n_datasets=("dataset_id", "nunique"),
        n_stages=("development_stage", "nunique")
    )
)

display(donor_check.describe())

print(
    "Donors with multiple diseases:",
    (donor_check.n_diseases > 1).sum()
)

print(
    "Donors represented in multiple datasets:",
    (donor_check.n_datasets > 1).sum()
)

/var/folders/_t/qn8rp_j9133d3727mgjxqfb40000gn/T/ipykernel_99787/1417499201.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs.groupby("donor_id")


,n_cells,n_diseases,n_sexes,n_datasets,n_stages
count,11633.000000,11633.000000,11633.000000,11633.000000,11633.000000
mean,90.553941,0.005416,0.005416,0.005416,0.005416
std,1242.854653,0.073395,0.073395,0.073395,0.073395
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000
max,24539.000000,1.000000,1.000000,1.000000,1.000000


Donors with multiple diseases: 0
Donors represented in multiple datasets: 0


In [13]:
donor_check = (
    obs.groupby("donor_id", observed=True)
    .agg(
        n_cells=("soma_joinid", "size"),
        n_diseases=("disease", "nunique"),
        n_sexes=("sex", "nunique"),
        n_datasets=("dataset_id", "nunique"),
        n_stages=("development_stage", "nunique")
    )
)

display(donor_check.describe())

print(
    "Number of donors:",
    len(donor_check)
)

print(
    "Donors with multiple diseases:",
    (donor_check["n_diseases"] > 1).sum()
)

print(
    "Donors with multiple sexes:",
    (donor_check["n_sexes"] > 1).sum()
)

print(
    "Donors represented in multiple datasets:",
    (donor_check["n_datasets"] > 1).sum()
)

print(
    "Donors with multiple development stages:",
    (donor_check["n_stages"] > 1).sum()
)

,n_cells,n_diseases,n_sexes,n_datasets,n_stages
count,63.000000,63.0,63.0,63.0,63.0
mean,16720.857143,1.0,1.0,1.0,1.0
std,2691.722610,0.0,0.0,0.0,0.0
min,6632.000000,1.0,1.0,1.0,1.0
25%,15202.000000,1.0,1.0,1.0,1.0
50%,16748.000000,1.0,1.0,1.0,1.0
75%,18292.000000,1.0,1.0,1.0,1.0
max,24539.000000,1.0,1.0,1.0,1.0


Number of donors: 63
Donors with multiple diseases: 0
Donors with multiple sexes: 0
Donors represented in multiple datasets: 0
Donors with multiple development stages: 0


In [14]:
# # import os, time, numpy as np, pandas as pd, scanpy as sc, anndata as ad, cellxgene_census

# parts = os.path.join(path, 'iha_parts')
# os.makedirs(parts, exist_ok = True)

# obs_cols = ['soma_joinid', 'cell_type', 'donor_id', 'assay', 'sex', 'development_stage', 'dataset_id', 'disease']

# test_1donor = True      # сначала True: прочитает одного донора и покажет время; потом False

# def joinids(a):
#     return (a.obs['soma_joinid'] if 'soma_joinid' in a.obs else pd.Series(a.obs_names, index = a.obs_names)).astype(int).to_numpy()

# donors = sub.donor_id.unique().tolist()
# todo = donors[:1] if test_1donor else donors
# t_all = time.time()
# with cellxgene_census.open_soma(census_version = '2025-11-08') as census:
#     for k, d in enumerate(todo, 1):
#         path = os.path.join(parts, f'{d}.h5ad')
#         if os.path.exists(path):
#             print(f'[{k}/{len(todo)}] {d}: уже есть', flush = True); continue
#         ids = np.sort(sub.loc[sub.donor_id == d, 'soma_joinid'].to_numpy())
#         t0 = time.time()
#         a = cellxgene_census.get_anndata(census, organism = 'Homo sapiens', obs_coords = ids, obs_column_names = obs_cols)
#         assert set(joinids(a)) == set(ids), f'{d}: прочитались не те клетки'
#         a.write_h5ad(path)
#         print(f'[{k}/{len(todo)}] {d}: {a.n_obs} клеток за {time.time()-t0:.0f} с (всего {time.time()-t_all:.0f} с)', flush = True)

In [15]:
# Проверяем все доступные столбцы obs

print("Number of columns:", len(obs.columns))

for col in obs.columns:
    print(col)

Number of columns: 11
soma_joinid
dataset_id
donor_id
cell_type
disease
sex
development_stage
tissue
tissue_general
assay
is_primary_data


In [16]:
interesting_columns = [
    col for col in obs.columns
    if any(
        key in col.lower()
        for key in [
            "subject",
            "sample",
            "specimen",
            "age",
            "donor",
            "guid"
        ]
    )
]

print(interesting_columns)

['donor_id', 'development_stage']


In [17]:
import pandas as pd

clinical = pd.read_csv(f"{path}/human_immune_health_atlas_metadata_clinical_labs.csv"
)

print("Shape:", clinical.shape)

print("\nColumns:")
print(clinical.columns.tolist())

display(clinical.head())

Shape: (108, 74)

Columns:
['cohort.cohortGuid', 'subject.subjectGuid', 'sample.sampleKitGuid', 'subject.biologicalSex', 'subject.birthYear', 'subject.ageAtFirstDraw', 'subject.ageGroup', 'subject.race', 'subject.ethnicity', 'sample.visitName', 'sample.visitDetails', 'sample.drawYear', 'sample.subjectAgeAtDraw', 'sample.daysSinceFirstVisit', 'sample.diseaseStatesRecordedAtVisit', 'am.bmi', 'am.height', 'am.weight', 'chem.alt', 'chem.albumin', 'chem.alkaline_phosphatase', 'chem.ast', 'chem.t_bili', 'chem.bun', 'chem.calcium', 'chem.co2', 'chem.cl', 'chem.creatinine', 'chem.egfr_aa', 'chem.egfr_non_aa', 'chem.globin', 'chem.glucose', 'chem.ldh', 'chem.magnesium', 'chem.phosphate', 'chem.potassium', 'chem.protein', 'chem.sodium', 'bc.perc_basophils', 'bc.perc_eosinophils', 'bc.perc_lymphocytes', 'bc.perc_monocytes', 'bc.perc_neutrophils', 'bc.basophil_count', 'bc.eosinophil_count', 'bc.lymphocyte_count', 'bc.monocyte_count', 'bc.neutrophil_count', 'bc.hematocrit', 'bc.hemoglobin', 'bc.mch

,cohort.cohortGuid,subject.subjectGuid,sample.sampleKitGuid,subject.biologicalSex,subject.birthYear,subject.ageAtFirstDraw,subject.ageGroup,subject.race,subject.ethnicity,sample.visitName,...,infl.rf_iga_result,infl.rf_igm_interpretation,infl.rf_igm_result,infl.esr,lip.cholesterol_hdl,lip.cholesterol_ldl,lip.cholesterol_non_hdl,lip.cholesterol_total,lip.chlesterol_hdl_ratio,lip.triglycerides
0,BR1,BR1001,KT00001,Female,1987,32,Young Adult,Caucasian,Non-Hispanic origin,Flu Year 1 Day 0,...,0.0,Negative,5.902,2.0,65.0,85.0,104.0,169.0,2.6,94.0
1,BR1,BR1002,KT00002,Male,1991,28,Young Adult,Caucasian,Non-Hispanic origin,Flu Year 1 Day 0,...,NaN,NaN,NaN,2.0,56.0,72.0,99.0,155.0,2.8,197.0
2,BR1,BR1003,KT00003,Female,1989,30,Young Adult,Caucasian,Non-Hispanic origin,Flu Year 1 Day 0,...,NaN,NaN,NaN,2.0,76.0,112.0,126.0,202.0,2.7,57.0
3,BR1,BR1004,KT00004,Male,1989,30,Young Adult,Caucasian,Non-Hispanic origin,Flu Year 1 Day 0,...,NaN,NaN,NaN,2.0,60.0,63.0,76.0,136.0,2.3,47.0
4,BR1,BR1005,KT00006,Female,1992,27,Young Adult,Caucasian,Non-Hispanic origin,Flu Year 1 Day 0,...,NaN,NaN,NaN,2.0,50.0,79.0,102.0,152.0,3.0,130.0


In [18]:
age_columns = [
    "subject.subjectGuid",
    "subject.ageAtFirstDraw",
    "sample.sampleKitGuid",
    "sample.subjectAgeAtDraw",
    "sample.drawYear"
]

display(
    clinical[age_columns].head(10)
)

print("\nUnique subjects:",
      clinical["subject.subjectGuid"].nunique())

print("\nUnique samples:",
      clinical["sample.sampleKitGuid"].nunique())

print("\nAge distribution:")

print(
    clinical["sample.subjectAgeAtDraw"].describe()
)

,subject.subjectGuid,subject.ageAtFirstDraw,sample.sampleKitGuid,sample.subjectAgeAtDraw,sample.drawYear
0,BR1001,32,KT00001,32,2019
1,BR1002,28,KT00002,28,2019
2,BR1003,30,KT00003,30,2019
3,BR1004,30,KT00004,30,2019
4,BR1005,27,KT00006,27,2019
5,BR1006,31,KT00010,31,2019
6,BR1008,34,KT00012,34,2019
7,BR1007,30,KT00014,30,2019
8,BR2003,59,KT00015,59,2019
9,BR2002,57,KT00016,57,2019



Unique subjects: 108

Unique samples: 108

Age distribution:
count    108.000000
mean      39.546296
std       17.913282
min       11.000000
25%       28.000000
50%       34.000000
75%       58.000000
max       65.000000
Name: sample.subjectAgeAtDraw, dtype: float64


In [19]:
# Prepare clinical metadata for merging

clinical_age = clinical[
    [
        "subject.subjectGuid",
        "sample.sampleKitGuid",
        "sample.subjectAgeAtDraw",
        "subject.ageAtFirstDraw",
        "subject.biologicalSex"
    ]
].copy()

# Rename columns for convenience

clinical_age = clinical_age.rename(
    columns={
        "subject.subjectGuid": "donor_id",
        "sample.sampleKitGuid": "sample_id",
        "sample.subjectAgeAtDraw": "age",
        "subject.ageAtFirstDraw": "age_at_first_draw",
        "subject.biologicalSex": "clinical_sex"
    }
)

# Check uniqueness before merging

print(
    "Unique clinical donor IDs:",
    clinical_age["donor_id"].nunique()
)

print(
    "Duplicate donor IDs:",
    clinical_age["donor_id"].duplicated().sum()
)

# Merge with Census observations

obs = obs.merge(
    clinical_age,
    on="donor_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Check merge results

print("\nMerge results:")
print(obs["_merge"].value_counts())

print(
    "\nDonors with missing age:",
    obs.loc[obs["age"].isna(), "donor_id"].nunique()
)

print(
    "\nUnique donors after merging:",
    obs["donor_id"].nunique()
)

Unique clinical donor IDs: 108
Duplicate donor IDs: 0

Merge results:
_merge
both          1053414
left_only           0
right_only          0
Name: count, dtype: int64

Donors with missing age: 0

Unique donors after merging: 63


In [20]:
import numpy as np

obs["age_group"] = pd.cut(
    obs["age"],
    bins=[-np.inf, 18, 45, np.inf],
    labels=["<18", "18-45", ">45"],
    right=False
)

In [21]:
donor_ages = (
    obs.groupby("donor_id", observed=True)
    .agg(
        age=("age", "first"),
        age_group=("age_group", "first"),
        n_cells=("soma_joinid", "size"),
        sex=("sex", "first")
    )
    .reset_index()
)

print("Donors per age group:")

display(
    donor_ages.groupby(
        "age_group",
        observed=True
    ).agg(
        n_donors=("donor_id", "nunique"),
        mean_age=("age", "mean"),
        min_age=("age", "min"),
        max_age=("age", "max"),
        mean_cells=("n_cells", "mean")
    )
)

Donors per age group:


,n_donors,mean_age,min_age,max_age,mean_cells
age_group,,,,,
<18,13,11.230769,11,12,15940.384615
18-45,29,30.206897,25,35,17178.413793
>45,21,58.714286,55,65,16572.142857


In [22]:
display(
    donor_ages.sort_values(
        ["age_group", "age"]
    )
)

,donor_id,age,age_group,n_cells,sex
50,UP1008,11,<18,20902,female
51,UP1011,11,<18,14645,female
52,UP1012,11,<18,16176,female
53,UP1014,11,<18,20613,female
54,UP1015,11,<18,18620,male
...,...,...,...,...,...
36,BR2023,61,>45,16175,male
39,BR2028,62,>45,20762,male
33,BR2020,63,>45,15328,male
42,BR2034,63,>45,16734,female


In [23]:
# Count cells per donor and cell type

donor_celltype_counts = (
    obs.groupby(
        ["donor_id", "cell_type"],
        observed=True
    )
    .size()
    .reset_index(name="n_cells")
)

# Count cell types per donor before applying minimum threshold

donor_type_summary = (
    donor_celltype_counts
    .groupby("donor_id", observed=True)
    .agg(
        n_cell_types=("cell_type", "nunique"),
        total_cells=("n_cells", "sum")
    )
)

# Count cell types with at least 10 cells

donor_type_summary["n_cell_types_min10"] = (
    donor_celltype_counts[
        donor_celltype_counts["n_cells"] >= 10
    ]
    .groupby("donor_id", observed=True)
    .size()
)

donor_type_summary["n_cell_types_min10"] = (
    donor_type_summary["n_cell_types_min10"]
    .fillna(0)
    .astype(int)
)

display(
    donor_type_summary.describe()
)

,n_cell_types,total_cells,n_cell_types_min10
count,63.000000,63.000000,63.000000
mean,37.158730,16720.857143,30.095238
std,0.745012,2691.722610,1.802456
min,35.000000,6632.000000,25.000000
25%,37.000000,15202.000000,29.000000
50%,37.000000,16748.000000,30.000000
75%,38.000000,18292.000000,31.000000
max,38.000000,24539.000000,34.000000


In [24]:
print(
    "Donors with at least 8 cell types:",
    (donor_type_summary["n_cell_types"] >= 8).sum()
)

print(
    "Donors with at least 8 cell types (minimum 10 cells):",
    (donor_type_summary["n_cell_types_min10"] >= 8).sum()
)

print(
    "\nDonors failing the 8 cell types criterion:"
)

display(
    donor_type_summary[
        donor_type_summary["n_cell_types_min10"] < 8
    ]
)

Donors with at least 8 cell types: 63
Donors with at least 8 cell types (minimum 10 cells): 63

Donors failing the 8 cell types criterion:


,n_cell_types,total_cells,n_cell_types_min10
donor_id,,,


In [25]:
donor_type_summary = donor_type_summary.join(
    donor_ages.set_index("donor_id")[["age", "age_group"]]
)

age_eligibility = (
    donor_type_summary
    .groupby("age_group", observed=True)
    .agg(
        total_donors=("n_cell_types_min10", "size"),
        eligible_donors=(
            "n_cell_types_min10",
            lambda x: (x >= 8).sum()
        ),
        mean_cell_types=("n_cell_types_min10", "mean"),
        min_cell_types=("n_cell_types_min10", "min"),
        max_cell_types=("n_cell_types_min10", "max")
    )
)

display(age_eligibility)

,total_donors,eligible_donors,mean_cell_types,min_cell_types,max_cell_types
age_group,,,,,
<18,13,13,29.538462,26,32
18-45,29,29,30.448276,26,34
>45,21,21,29.952381,25,33


In [26]:
# Prepare eligible donors

eligible_donors = donor_type_summary.reset_index()

eligible_donors = eligible_donors[
    (eligible_donors["total_cells"] >= 2000) &
    (eligible_donors["n_cell_types_min10"] >= 8)
].copy()

print("Eligible donors:", len(eligible_donors))

# Random selection: 8 donors per age group

selected_donors = pd.concat(
    [
        group.sample(n=8, random_state=0)
        for age_group, group in eligible_donors.groupby(
            "age_group",
            observed=True
        )
    ],
    ignore_index=True
)

selected_donors = selected_donors.sort_values(
    ["age_group", "age"]
).reset_index(drop=True)

display(selected_donors)

print("\nDonors per age group:")
print(
    selected_donors["age_group"].value_counts()
)

Eligible donors: 63


,donor_id,n_cell_types,total_cells,n_cell_types_min10,age,age_group
0,UP1021,37,14124,29,11,<18
1,UP1015,37,18620,31,11,<18
2,UP1031,38,17706,29,11,<18
3,UP1012,36,16176,26,11,<18
4,UP1011,37,14645,29,11,<18
5,UP1023,38,13291,30,11,<18
6,UP1032,37,16345,30,12,<18
7,UP1024,37,6632,29,12,<18
8,BR1056,37,15475,31,25,18-45
9,BR1057,37,16962,29,25,18-45



Donors per age group:
age_group
<18      8
18-45    8
>45      8
Name: count, dtype: int64


In [27]:
# Keep only selected donors

selected_obs = obs[
    obs["donor_id"].isin(
        selected_donors["donor_id"]
    )
].copy()

# Count cells per donor × cell type

group_counts = (
    selected_obs.groupby(
        ["donor_id", "cell_type"],
        observed=True
    )
    .size()
)

# Keep combinations with at least 10 cells

valid_groups = group_counts[
    group_counts >= 10
].index

selected_obs = (
    selected_obs.set_index(
        ["donor_id", "cell_type"]
    )
    .loc[valid_groups]
    .reset_index()
)

# Sample maximum 100 cells per donor × cell type

sampled_parts = []

for (donor, cell_type), group in selected_obs.groupby(
    ["donor_id", "cell_type"],
    observed=True
):

    n = min(len(group), 100)

    sampled_parts.append(
        group.sample(
            n=n,
            random_state=0
        )
    )

sampled_obs = pd.concat(
    sampled_parts,
    ignore_index=True
)

print("Final number of cells:", len(sampled_obs))

print(
    "Final number of donors:",
    sampled_obs["donor_id"].nunique()
)

print(
    "Final number of cell types:",
    sampled_obs["cell_type"].nunique()
)

Final number of cells: 54769
Final number of donors: 24
Final number of cell types: 36


In [28]:
# Cells per donor

final_donor_stats = (
    sampled_obs.groupby(
        ["donor_id", "age_group"],
        observed=True
    )
    .agg(
        n_cells=("soma_joinid", "size"),
        n_cell_types=("cell_type", "nunique")
    )
)

display(final_donor_stats)

# Cells per donor × cell type

final_group_counts = (
    sampled_obs.groupby(
        ["donor_id", "cell_type"],
        observed=True
    )
    .size()
)

print(
    "\nMinimum cells per donor:",
    final_donor_stats["n_cells"].min()
)

print(
    "Minimum cell types per donor:",
    final_donor_stats["n_cell_types"].min()
)

print(
    "Maximum cells per donor × cell type:",
    final_group_counts.max()
)

print(
    "Minimum cells per donor × cell type:",
    final_group_counts.min()
)

,,n_cells,n_cell_types
donor_id,age_group,,
BR1003,18-45,2416,33
BR1018,18-45,2246,29
BR1021,18-45,2126,30
BR1042,18-45,2142,30
BR1052,18-45,2317,29
BR1056,18-45,2373,31
BR1057,18-45,2172,29
BR1058,18-45,2373,32
BR2008,>45,2238,30



Minimum cells per donor: 1964
Minimum cell types per donor: 26
Maximum cells per donor × cell type: 100
Minimum cells per donor × cell type: 10


In [29]:
# Save sampled cell IDs and metadata

sample_ids = sampled_obs[
    [
        "soma_joinid",
        "dataset_id",
        "donor_id",
        "cell_type",
        "age",
        "age_group",
        "sex"
    ]
].copy()

sample_ids.to_csv(
    f"{path}/iha_sample_ids.csv",
    index=False
)

print("Saved:", f"{path}/iha_sample_ids.csv")
print("Number of sampled cells:", len(sample_ids))

Saved: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_sample_ids.csv
Number of sampled cells: 54769


In [30]:
# Calculate expected final cell count per donor
# after applying min 10 and max 100 cells per cell type

candidate_counts = donor_celltype_counts.copy()

candidate_counts["sampled_cells"] = np.where(
    candidate_counts["n_cells"] >= 10,
    candidate_counts["n_cells"].clip(upper=100),
    0
)

candidate_final = (
    candidate_counts
    .groupby("donor_id", observed=True)
    .agg(
        final_cells=("sampled_cells", "sum"),
        n_cell_types=("sampled_cells",
                      lambda x: (x > 0).sum())
    )
    .reset_index()
)

candidate_final = candidate_final.merge(
    donor_ages[
        ["donor_id", "age", "age_group", "n_cells"]
    ],
    on="donor_id",
    how="left"
)

display(
    candidate_final.sort_values(
        ["age_group", "final_cells"]
    )
)

,donor_id,final_cells,n_cell_types,age,age_group,n_cells
58,UP1024,1964,29,12,<18,6632
62,UP1034,2116,28,12,<18,18018
59,UP1025,2133,31,11,<18,13282
61,UP1032,2208,30,12,<18,16345
51,UP1011,2222,29,11,<18,14645
...,...,...,...,...,...,...
47,BR2042,2309,31,55,>45,16943
45,BR2039,2372,32,55,>45,18063
39,BR2028,2418,31,62,>45,20762
37,BR2025,2433,33,55,>45,24539


In [31]:
print(
    candidate_final.groupby(
        "age_group",
        observed=True
    ).agg(
        total_donors=("donor_id", "count"),
        donors_above_2000=(
            "final_cells",
            lambda x: (x >= 2000).sum()
        ),
        min_final_cells=("final_cells", "min"),
        max_final_cells=("final_cells", "max")
    )
)

           total_donors  donors_above_2000  min_final_cells  max_final_cells
age_group                                                                   
<18                  13                 12             1964             2578
18-45                29                 29             2093             2520
>45                  21                 19             1787             2438


In [32]:
eligible_final = candidate_final[
    (candidate_final["n_cells"] >= 2000) &
    (candidate_final["n_cell_types"] >= 8) &
    (candidate_final["final_cells"] >= 2000)
].copy()

selected_donors = pd.concat(
    [
        group.sample(n=8, random_state=0)
        for _, group in eligible_final.groupby(
            "age_group",
            observed=True
        )
    ],
    ignore_index=True
)

display(
    selected_donors[
        [
            "donor_id",
            "age",
            "age_group",
            "final_cells",
            "n_cell_types"
        ]
    ].sort_values(
        ["age_group", "age"]
    )
)

,donor_id,age,age_group,final_cells,n_cell_types
0,UP1021,11,<18,2286,29
2,UP1015,11,<18,2312,31
4,UP1012,11,<18,2275,26
5,UP1025,11,<18,2133,31
6,UP1011,11,<18,2222,29
7,UP1023,11,<18,2578,30
1,UP1034,12,<18,2116,28
3,UP1032,12,<18,2208,30
13,BR1056,25,18-45,2373,31
15,BR1057,25,18-45,2172,29


In [33]:
# Select cells from the 24 selected donors
selected_obs = obs[
    obs["donor_id"].isin(selected_donors["donor_id"])
].copy()

# Count cells per donor and cell type
group_counts = (
    selected_obs
    .groupby(["donor_id", "cell_type"], observed=True)
    .size()
)

# Keep only donor × cell-type groups with at least 10 cells
valid_groups = group_counts[group_counts >= 10].index

selected_obs = (
    selected_obs
    .set_index(["donor_id", "cell_type"])
    .loc[valid_groups]
    .reset_index()
)

# Sample up to 100 cells per donor × cell type
sampled_parts = []

for (donor, cell_type), group in selected_obs.groupby(
    ["donor_id", "cell_type"], observed=True
):
    n = min(len(group), 100)

    sampled_parts.append(
        group.sample(n=n, random_state=0)
    )

sampled_obs = pd.concat(sampled_parts, ignore_index=True)

print("Total sampled cells:", len(sampled_obs))
print("Number of donors:", sampled_obs["donor_id"].nunique())
print("Number of cell types:", sampled_obs["cell_type"].nunique())

Total sampled cells: 53974
Number of donors: 24
Number of cell types: 36


In [34]:
# Summary per donor
final_donor_summary = (
    sampled_obs
    .groupby("donor_id", observed=True)
    .agg(
        n_cells=("soma_joinid", "size"),
        n_cell_types=("cell_type", "nunique"),
        age=("age", "first"),
        age_group=("age_group", "first")
    )
    .reset_index()
)

display(
    final_donor_summary
    .sort_values(["age_group", "age", "donor_id"])
)

# Counts per donor × cell type
final_group_counts = (
    sampled_obs
    .groupby(["donor_id", "cell_type"], observed=True)
    .size()
)

print("Minimum cells per donor:",
      final_donor_summary["n_cells"].min())

print("Minimum cell types per donor:",
      final_donor_summary["n_cell_types"].min())

print("Minimum cells per donor × cell type:",
      final_group_counts.min())

print("Maximum cells per donor × cell type:",
      final_group_counts.max())

print("Duplicate soma_joinid values:",
      sampled_obs["soma_joinid"].duplicated().sum())

print("\nDonors per age group:")
print(
    final_donor_summary
    .groupby("age_group", observed=True)
    .size()
)

,donor_id,n_cells,n_cell_types,age,age_group
16,UP1011,2222,29,11,<18
17,UP1012,2275,26,11,<18
18,UP1015,2312,31,11,<18
19,UP1021,2286,29,11,<18
20,UP1023,2578,30,11,<18
21,UP1025,2133,31,11,<18
22,UP1032,2208,30,12,<18
23,UP1034,2116,28,12,<18
5,BR1056,2373,31,25,18-45
6,BR1057,2172,29,25,18-45


Minimum cells per donor: 2116
Minimum cell types per donor: 26
Minimum cells per donor × cell type: 10
Maximum cells per donor × cell type: 100
Duplicate soma_joinid values: 0

Donors per age group:
age_group
<18      8
18-45    8
>45      8
dtype: int64


In [35]:
sample_ids = sampled_obs[
    [
        "soma_joinid",
        "dataset_id",
        "donor_id",
        "cell_type",
        "age",
        "age_group",
        "sex"
    ]
].copy()

output_path = (
    "/Users/artsvetkova/Studies/_JKU/3rd_semester/"
    "AI and Visualization/UE/iha_sample_ids.csv"
)

sample_ids.to_csv(output_path, index=False)

print(f"Saved {len(sample_ids)} cell IDs to:")
print(output_path)

Saved 53974 cell IDs to:
/Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_sample_ids.csv


In [36]:
import pandas as pd
import numpy as np
import scipy.sparse as sp

sample_ids = pd.read_csv(output_path)

print("Number of cells:", len(sample_ids))
print("Unique soma_joinid:", sample_ids["soma_joinid"].nunique())
print("Unique donors:", sample_ids["donor_id"].nunique())

assert sample_ids["soma_joinid"].is_unique
assert sample_ids["donor_id"].nunique() == 24

obs_coords = sample_ids["soma_joinid"].astype("int64").tolist()

Number of cells: 53974
Unique soma_joinid: 53974
Unique donors: 24


In [37]:
sample_ids = pd.read_csv(output_path)

print("Cells:", len(sample_ids))
print("Donors:", sample_ids["donor_id"].nunique())

assert sample_ids["soma_joinid"].is_unique

Cells: 53974
Donors: 24


In [38]:
import cellxgene_census
import anndata as ad
import time

expression_path = os.path.join(
    base_dir,
    "iha_expression_raw.h5ad"
)

obs_coords = np.sort(
    sample_ids["soma_joinid"].to_numpy()
)

with cellxgene_census.open_soma(
    census_version=census_version
) as census:

    adata = cellxgene_census.get_anndata(
        census,
        organism="Homo sapiens",
        obs_coords=obs_coords
    )

print(adata)

/Users/artsvetkova/miniconda3/envs/sc_immune/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/artsvetkova/miniconda3/envs/sc_immune/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 53974 × 61497
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars'
    var: 'soma_joinid', 'feature_id', 'feature_name', 'feature_type', 'feature_length', 'nnz', 'n_measured_obs'


In [39]:
print("AnnData cells:", adata.n_obs)
print("Genes:", adata.n_vars)

assert adata.n_obs == len(sample_ids)
assert set(adata.obs["soma_joinid"]) == set(
    sample_ids["soma_joinid"]
)

AnnData cells: 53974
Genes: 61497


In [40]:
adata.write_h5ad(expression_path)

print("Saved:", expression_path)

Saved: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/iha_expression_raw.h5ad


In [41]:
import celltypist
from celltypist import models

models.download_models(
    model="Immune_All_Low.pkl"
)

/Users/artsvetkova/miniconda3/envs/sc_immune/lib/python3.10/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv
📂 Storing models in /Users/artsvetkova/.celltypist/data/models
💾 Total models to download: 1
⏩ Skipping [1/1]: Immune_All_Low.pkl (file exists)


In [42]:
models.models_description()

👉 Detailed model information can be found at `https://www.celltypist.org/models`


,model,description
0,Immune_All_Low.pkl,immune sub-populations combined from 20 tissue...
1,Immune_All_High.pkl,immune populations combined from 20 tissues of...
2,Adult_COVID19_PBMC.pkl,peripheral blood mononuclear cell types from C...
3,Adult_CynomolgusMacaque_Hippocampus.pkl,cell types from the hippocampus of adult cynom...
4,Adult_Human_MTG.pkl,cell types and subtypes (10x-based) from the a...
...,...,...
56,Nuclei_Lung_Airway.pkl,cell populations from snRNA-seq of five locati...
57,PaediatricAdult_COVID19_Airway.pkl,cell types in the airway of paediatric and adu...
58,PaediatricAdult_COVID19_PBMC.pkl,peripheral blood mononuclear cell types of pae...
59,Pan_Fetal_Human.pkl,stromal and immune populations from the human ...


In [43]:
models.models_path

'/Users/artsvetkova/.celltypist/data/models'

In [44]:
model = models.Model.load(
    model="Immune_All_Low.pkl"
)

print(model)

CellTypist model with 98 cell types and 6639 features
    date: 2022-07-16 00:20:42.927778
    details: immune sub-populations combined from 20 tissues of 18 studies
    source: https://doi.org/10.1126/science.abl5197
    version: v2
    cell types: Age-associated B cells, Alveolar macrophages, ..., pDC precursor
    features: A1BG, A2M, ..., ZYX


In [45]:
print(
    len(model.classifier.classes_)
)

print(
    model.classifier.classes_
)

98
['Age-associated B cells' 'Alveolar macrophages' 'B cells'
 'CD16+ NK cells' 'CD16- NK cells' 'CD8a/a' 'CD8a/b(entry)' 'CMP'
 'CRTAM+ gamma-delta T cells' 'Classical monocytes' 'Cycling B cells'
 'Cycling DCs' 'Cycling NK cells' 'Cycling T cells'
 'Cycling gamma-delta T cells' 'Cycling monocytes' 'DC' 'DC precursor'
 'DC1' 'DC2' 'DC3' 'Double-negative thymocytes'
 'Double-positive thymocytes' 'ELP' 'ETP' 'Early MK' 'Early erythroid'
 'Early lymphoid/T lymphoid' 'Endothelial cells' 'Epithelial cells'
 'Erythrocytes' 'Erythrophagocytic macrophages' 'Fibroblasts'
 'Follicular B cells' 'Follicular helper T cells' 'GMP'
 'Germinal center B cells' 'Granulocytes' 'HSC/MPP' 'Hofbauer cells' 'ILC'
 'ILC precursor' 'ILC1' 'ILC2' 'ILC3' 'Intermediate macrophages'
 'Intestinal macrophages' 'Kidney-resident macrophages' 'Kupffer cells'
 'Large pre-B cells' 'Late erythroid' 'MAIT cells' 'MEMP' 'MNP'
 'Macrophages' 'Mast cells' 'Megakaryocyte precursor'
 'Megakaryocyte-erythroid-mast cell progenit

In [46]:
adata = sc.read_h5ad(expression_path)

adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(
    adata,
    target_sum=10000
)

sc.pp.log1p(adata)

In [47]:
predictions = celltypist.annotate(
    adata,
    model="Immune_All_Low.pkl",
    majority_voting=False
)

🔬 Input data has 53974 cells and 61497 genes
🔗 Matching reference genes in the model


ValueError: 🛑 No features overlap with the model. Please provide gene symbols

In [49]:
print("Current var_names:")
print(adata.var_names[:10])

print("\nFeature names:")
print(adata.var["feature_name"].head(10))

Current var_names:
Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'], dtype='object')

Feature names:
0          LINC01409
1              NOC2L
2              PERM1
3    ENSG00000272512
4               HES4
5              ISG15
6               AGRN
7             RNF223
8           C1orf159
9           TNFRSF18
Name: feature_name, dtype: category
Categories (60028, object): ['5S_rRNA', '5_8S_rRNA', '7SK', 'A1BG', ..., 'hsa-mir-423', 'hsa-mir-1253', 'hsa-mir-8069-1', 'snoZ196']


In [50]:
# Используем gene symbols вместо Ensembl IDs
adata.var_names = adata.var["feature_name"].astype(str)

# Проверяем и устраняем дублирующиеся названия
adata.var_names_make_unique()

print(adata.var_names[:20])

Index(['LINC01409', 'NOC2L', 'PERM1', 'ENSG00000272512', 'HES4', 'ISG15',
       'AGRN', 'RNF223', 'C1orf159', 'TNFRSF18', 'TNFRSF4', 'SDF4', 'B3GALT6',
       'UBE2J2', 'ACAP3', 'PUSL1', 'INTS11', 'CPTP', 'DVL1', 'MXRA8'],
      dtype='object', name='feature_name')


In [51]:
from celltypist import models

model = models.Model.load(
    model="Immune_All_Low.pkl"
)

model_genes = set(model.classifier.features)
data_genes = set(adata.var_names)

overlap = model_genes.intersection(data_genes)

print("Genes in model:", len(model_genes))
print("Genes in dataset:", len(data_genes))
print("Overlapping genes:", len(overlap))
print(
    "Percentage of model genes covered:",
    round(len(overlap) / len(model_genes) * 100, 2),
    "%"
)

Genes in model: 6639
Genes in dataset: 61497
Overlapping genes: 6090
Percentage of model genes covered: 91.73 %


In [52]:
import re

ensembl_mask = adata.var_names.str.startswith("ENSG")

print("Ensembl IDs:", ensembl_mask.sum())
print("Gene symbols:", (~ensembl_mask).sum())

Ensembl IDs: 18922
Gene symbols: 42575


In [53]:
print(adata.X[:5, :10])

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 3 stored elements and shape (5, 10)>
  Coords	Values
  (0, 9)	1.788641333580017
  (2, 9)	1.8458267450332642
  (3, 5)	2.5390079021453857


In [54]:
import numpy as np
import scipy.sparse as sp

print("Matrix shape:", adata.shape)
print("Matrix dtype:", adata.X.dtype)
print("Sparse:", sp.issparse(adata.X))

print("\nLayers:")
print(list(adata.layers.keys()))

print("\nExpression values:")

if sp.issparse(adata.X):
    values = adata.X.data
else:
    values = np.asarray(adata.X).flatten()

print("Min:", values.min())
print("Max:", values.max())
print("Mean:", values.mean())

print(
    "Fraction of integer-like values:",
    np.mean(np.isclose(values, np.round(values)))
)

Matrix shape: (53974, 61497)
Matrix dtype: float32
Sparse: True

Layers:
['counts']

Expression values:
Min: 0.21055885
Max: 9.031468
Mean: 1.3813887
Fraction of integer-like values: 5.010530027293197e-06


In [55]:
print(
    adata.obs[
        ["raw_sum", "nnz", "raw_mean_nnz"]
    ].head()
)

print("\nCurrent matrix row sums:")

print(
    np.asarray(adata.X.sum(axis=1)).flatten()[:10]
)

   raw_sum   nnz  raw_mean_nnz
0   4015.0  1324      3.032477
1   8604.0  2838      3.031712
2   1875.0   841      2.229489
3  15428.0  3529      4.371777
4   1326.0   735      1.804082

Current matrix row sums:
[2124.036  3243.4275 1833.5353 3103.2788 1775.3644 1912.762  1830.51
 2586.413  2873.019  1641.3168]


In [56]:
import numpy as np
import scipy.sparse as sp

for name, matrix in [
    ("X", adata.X),
    ("counts", adata.layers["counts"])
]:

    print(f"\n--- {name} ---")

    print("Shape:", matrix.shape)
    print("Sparse:", sp.issparse(matrix))

    if sp.issparse(matrix):
        values = matrix.data
        row_sums = np.asarray(
            matrix.sum(axis=1)
        ).flatten()
    else:
        values = np.asarray(matrix).flatten()
        row_sums = np.asarray(
            matrix.sum(axis=1)
        ).flatten()

    print("Min:", values.min())
    print("Max:", values.max())
    print("Mean:", values.mean())

    print(
        "Integer-like:",
        np.mean(np.isclose(values, np.round(values)))
    )

    print("First 5 row sums:", row_sums[:5])


--- X ---
Shape: (53974, 61497)
Sparse: True
Min: 0.21055885
Max: 9.031468
Mean: 1.3813887
Integer-like: 5.010530027293197e-06
First 5 row sums: [2124.036  3243.4275 1833.5353 3103.2788 1775.3644]

--- counts ---
Shape: (53974, 61497)
Sparse: True
Min: 1.0
Max: 20999.0
Mean: 3.295568
Integer-like: 1.0
First 5 row sums: [ 4015.  8604.  1875. 15428.  1326.]


In [57]:
print(
    "Raw sums from Census:",
    adata.obs["raw_sum"].head().tolist()
)

Raw sums from Census: [4015.0, 8604.0, 1875.0, 15428.0, 1326.0]


In [58]:
predictions = celltypist.annotate(
    adata,
    model="Immune_All_Low.pkl",
    majority_voting=False
)

🔬 Input data has 53974 cells and 61497 genes
🔗 Matching reference genes in the model
🧬 6090 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!


In [59]:
predicted_labels = predictions.predicted_labels

print(predicted_labels.head())

              predicted_labels
0     Tcm/Naive helper T cells
1             Cycling NK cells
2  Tem/Effector helper T cells
3                          DC2
4        Trm cytotoxic T cells


In [60]:
print(
    predicted_labels["predicted_labels"]
    .value_counts()
)

predicted_labels
Tcm/Naive helper T cells     7529
Naive B cells                4863
Regulatory T cells           4282
CD16+ NK cells               3411
Tem/Trm cytotoxic T cells    3308
                             ... 
CMP                             1
Kupffer cells                   1
T(agonist)                      1
Mono-mac                        1
Pro-B cells                     1
Name: count, Length: 61, dtype: int64


In [61]:
print(
    "Cells:", len(predicted_labels)
)

print(
    "Predicted cell types:",
    predicted_labels["predicted_labels"].nunique()
)

Cells: 53974
Predicted cell types: 61


In [62]:
predicted_labels.to_csv(
    os.path.join(
        base_dir,
        "baseline_predictions.csv"
    )
)

predictions.decision_matrix.to_csv(
    os.path.join(
        base_dir,
        "baseline_decision_matrix.csv"
    )
)

In [ ]:
adata.var.index.name = "gene_symbol"

adata.write_h5ad(
    os.path.join(base_dir, "iha_expression_prepared.h5ad"),
    compression="gzip"
)


In [65]:
print(adata.var.index.name)
print(adata.var.columns.tolist())
print(adata.var_names[:5].tolist())

gene_symbol
['soma_joinid', 'feature_id', 'feature_name', 'feature_type', 'feature_length', 'nnz', 'n_measured_obs']
['LINC01409', 'NOC2L', 'PERM1', 'ENSG00000272512', 'HES4']


In [66]:
predictions = celltypist.annotate(
    adata,
    model="Immune_All_Low.pkl",
    majority_voting=False
)

🔬 Input data has 53974 cells and 61497 genes
🔗 Matching reference genes in the model
🧬 6090 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
